In [1]:
from bs4 import BeautifulSoup, Comment
import pandas as pd

import requests
import time
import csv
from io import StringIO

import sys
from pathlib import Path

pd.set_option('display.max_columns', None)

ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
     ROOT = ROOT.parent
src_path = str(ROOT/"src")
if src_path not in sys.path:
    sys.path.insert(0,src_path)

from config import START_YEAR, END_YEAR, BASE_URL, HTTP_HEADER
from utils import scrape_per_game_tables

### [Basketball-Reference Bot Policy](https://www.sports-reference.com/bot-traffic.html)

Basketball reference enforces a 1 day block if you are caught accessing the site more than 20 times a second.  All cells that run loops which collect the data should have `time.sleep(x)` in the loop where x $\geq3$.  In what follows we use `time.sleep(3.3)` for good measure.

In [9]:
scrape_per_game_tables(START_YEAR,END_YEAR)

Scraping data for the 1980-1981 season...
Adding team win/losses...
Scraping data for the 1981-1982 season...
Adding team win/losses...
Scraping data for the 1982-1983 season...
Adding team win/losses...
Scraping data for the 1983-1984 season...
Adding team win/losses...
Scraping data for the 1984-1985 season...
Adding team win/losses...
Scraping data for the 1985-1986 season...
Adding team win/losses...
Scraping data for the 1986-1987 season...
Adding team win/losses...
Scraping data for the 1987-1988 season...
Adding team win/losses...
Scraping data for the 1988-1989 season...
Adding team win/losses...
Scraping data for the 1989-1990 season...
Adding team win/losses...
Scraping data for the 1990-1991 season...
Adding team win/losses...
Scraping data for the 1991-1992 season...
Adding team win/losses...
Scraping data for the 1992-1993 season...
Adding team win/losses...
Scraping data for the 1993-1994 season...
Adding team win/losses...
Scraping data for the 1994-1995 season...
Adding

when we import the datafram from the csv it is important to add a comment keyword or it will mislabel the columns using the descriptive comment instead of the names.

Example for the 1980-81 season

In [12]:
df = pd.read_csv(str(ROOT)+'/data/1980-1981_season/per_game_stats_1980-1981.csv',comment='#')

In [16]:
df.head()

,Year,Wins,Losses,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS
0,1980,28.0,54.0,Adrian Dantley,25.0,UTA,SF,80.0,NaN,42.7,11.4,20.3,0.559,0.0,0.1,0.286,11.3,20.3,0.560,0.559,7.9,9.8,0.806,2.4,4.0,6.4,4.0,1.4,0.2,3.5,3.1,30.7
1,1980,40.0,42.0,Moses Malone,25.0,HOU,C,80.0,NaN,40.6,10.1,19.3,0.522,0.0,0.0,0.333,10.1,19.3,0.522,0.522,7.6,10.1,0.757,5.9,8.8,14.8,1.8,1.0,1.9,3.9,2.8,27.8
2,1980,52.0,30.0,George Gervin,28.0,SAS,SG,82.0,82.0,33.7,10.4,21.1,0.492,0.1,0.4,0.257,10.3,20.7,0.496,0.494,6.2,7.6,0.826,1.5,3.6,5.1,3.2,1.1,0.7,3.1,2.6,27.1
3,1980,54.0,28.0,Kareem Abdul-Jabbar,33.0,LAL,C,80.0,NaN,37.2,10.5,18.2,0.574,0.0,0.0,0.000,10.5,18.2,0.574,0.574,5.3,6.9,0.766,2.5,7.8,10.3,3.4,0.7,2.9,3.1,3.1,26.2
4,1980,37.0,45.0,David Thompson,26.0,DEN,SG,77.0,NaN,34.0,9.5,18.8,0.506,0.1,0.5,0.256,9.4,18.3,0.513,0.509,6.4,8.0,0.795,1.4,2.3,3.7,3.0,0.7,0.8,3.2,3.0,25.5


Note that players who have been traded during the season show up multiple times.  One for each team as well as an aggregate row where the team is listed as "#TM" and # is the number of teams they have been on. We have made the team wins/losses for these rows NaNs.

In [14]:
df.loc[df['Team']=='2TM']

,Year,Wins,Losses,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS
85,1980,NaN,NaN,Geoff Huston,23.0,2TM,PG,81.0,NaN,30.0,5.7,11.6,0.489,0.0,0.1,0.200,5.7,11.6,0.491,0.490,1.9,2.6,0.708,0.6,1.1,1.7,4.9,0.7,0.1,2.2,1.8,13.2
93,1980,NaN,NaN,Bill Robinzine,28.0,2TM,PF,78.0,NaN,25.8,5.0,10.6,0.475,0.0,0.1,0.167,5.0,10.5,0.477,0.475,2.8,3.6,0.776,2.2,4.7,6.8,1.5,1.0,0.1,2.4,3.5,12.9
99,1980,NaN,NaN,Wes Matthews,21.0,2TM,PG,79.0,NaN,28.7,4.9,9.9,0.494,0.1,0.3,0.238,4.8,9.6,0.501,0.497,2.6,3.2,0.802,0.6,1.2,1.8,5.2,1.4,0.2,3.3,3.1,12.4
109,1980,NaN,NaN,Don Collins,22.0,2TM,SF,81.0,NaN,22.8,4.4,10.0,0.444,0.0,0.1,0.000,4.4,9.9,0.447,0.444,2.6,3.4,0.776,1.6,1.7,3.3,2.3,1.3,0.3,2.1,3.2,11.5
125,1980,NaN,NaN,Bob McAdoo,29.0,2TM,PF,16.0,NaN,20.1,4.3,9.8,0.433,0.0,0.1,0.000,4.3,9.8,0.436,0.433,1.8,2.6,0.707,1.1,3.1,4.2,1.9,1.1,0.8,2.0,2.4,10.3
128,1980,NaN,NaN,Dave Robisch,31.0,2TM,C,84.0,NaN,25.2,3.9,8.8,0.446,0.0,0.0,NaN,3.9,8.8,0.446,0.446,2.4,2.9,0.810,1.9,4.1,5.9,2.1,0.4,0.4,1.0,2.1,10.2
133,1980,NaN,NaN,Richard Washington,25.0,2TM,PF,80.0,NaN,22.7,4.3,9.3,0.455,0.0,0.0,0.500,4.2,9.3,0.455,0.456,1.5,2.0,0.748,2.0,3.7,5.7,1.6,0.6,0.8,1.6,3.5,10.0
140,1980,NaN,NaN,Ollie Mack,23.0,2TM,SG,65.0,NaN,25.9,4.3,9.3,0.460,0.0,0.1,0.000,4.3,9.2,0.467,0.460,1.2,1.9,0.640,1.4,2.1,3.5,2.5,0.9,0.1,1.1,1.8,9.8
144,1980,NaN,NaN,Billy McKinney,25.0,2TM,PG,84.0,NaN,25.8,3.9,7.7,0.507,0.0,0.1,0.167,3.9,7.5,0.513,0.509,1.9,2.2,0.862,0.4,1.8,2.2,4.3,1.2,0.1,1.9,2.8,9.7
158,1980,NaN,NaN,Joe Hassett,25.0,2TM,SG,41.0,NaN,17.4,3.5,8.3,0.421,1.3,3.8,0.340,2.2,4.5,0.489,0.499,0.4,0.5,0.810,0.6,1.1,1.7,1.8,0.3,0.0,0.5,1.6,8.7


Now we import all of the data into a single pandas dataframe

In [47]:
dft = pd.DataFrame()
for d in sorted(Path(str(ROOT)+'/data').iterdir()):
    #print(d.name)
    for f in Path(str(ROOT)+'/data/'+d.name).iterdir():
        #print(f.name)
        filepath = str(ROOT) + f'/data/{d.name}/{f.name}'
        df_tmp = pd.read_csv(filepath,encoding='utf-8',comment='#')
        #print(len(df_tmp))
        dft = pd.concat([dft,df_tmp],ignore_index=True)
dft

,Year,Wins,Losses,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS
0,1980,28.0,54.0,Adrian Dantley,25.0,UTA,SF,80.0,NaN,42.7,11.4,20.3,0.559,0.0,0.1,0.286,11.3,20.3,0.560,0.559,7.9,9.8,0.806,2.4,4.0,6.4,4.0,1.4,0.2,3.5,3.1,30.7
1,1980,40.0,42.0,Moses Malone,25.0,HOU,C,80.0,NaN,40.6,10.1,19.3,0.522,0.0,0.0,0.333,10.1,19.3,0.522,0.522,7.6,10.1,0.757,5.9,8.8,14.8,1.8,1.0,1.9,3.9,2.8,27.8
2,1980,52.0,30.0,George Gervin,28.0,SAS,SG,82.0,82.0,33.7,10.4,21.1,0.492,0.1,0.4,0.257,10.3,20.7,0.496,0.494,6.2,7.6,0.826,1.5,3.6,5.1,3.2,1.1,0.7,3.1,2.6,27.1
3,1980,54.0,28.0,Kareem Abdul-Jabbar,33.0,LAL,C,80.0,NaN,37.2,10.5,18.2,0.574,0.0,0.0,0.000,10.5,18.2,0.574,0.574,5.3,6.9,0.766,2.5,7.8,10.3,3.4,0.7,2.9,3.1,3.1,26.2
4,1980,37.0,45.0,David Thompson,26.0,DEN,SG,77.0,NaN,34.0,9.5,18.8,0.506,0.1,0.5,0.256,9.4,18.3,0.513,0.509,6.4,8.0,0.795,1.4,2.3,3.7,3.0,0.7,0.8,3.2,3.0,25.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24987,2025,42.0,40.0,Jahmyl Telfort,24.0,LAC,PF,8.0,0.0,4.0,0.0,0.5,0.000,0.0,0.5,0.000,0.0,0.0,NaN,0.000,0.1,0.5,0.250,0.0,0.4,0.4,0.1,0.1,0.0,0.1,0.6,0.1
24988,2025,52.0,30.0,Darius Brown II,26.0,CLE,SG,1.0,0.0,3.0,0.0,1.0,0.000,0.0,0.0,NaN,0.0,1.0,0.000,0.000,0.0,0.0,NaN,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
24989,2025,31.0,51.0,Noa Essengue,19.0,CHI,PF,2.0,0.0,3.0,0.0,1.5,0.000,0.0,1.0,0.000,0.0,0.5,0.000,0.000,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.5,0.0
24990,2025,53.0,29.0,Tosan Evbuomwan,24.0,NYK,SF,5.0,0.0,1.6,0.0,0.2,0.000,0.0,0.0,NaN,0.0,0.2,0.000,0.000,0.0,0.0,NaN,0.2,0.2,0.4,0.0,0.0,0.0,0.0,0.2,0.0
